# E-commerce Business Analytics Dashboard

**Comprehensive analysis of e-commerce performance metrics and business insights**

---

## Table of Contents

1. [Business Objectives & Configuration](#1-business-objectives--configuration)
2. [Data Dictionary](#2-data-dictionary)
3. [Data Loading & Validation](#3-data-loading--validation)
4. [Revenue & Growth Analysis](#4-revenue--growth-analysis)
5. [Product Performance Analysis](#5-product-performance-analysis)
6. [Geographic Revenue Analysis](#6-geographic-revenue-analysis)
7. [Customer Experience Analysis](#7-customer-experience-analysis)
8. [Executive Summary & Key Insights](#8-executive-summary--key-insights)

---

## 1. Business Objectives & Configuration

### Key Business Questions
- How did our revenue perform in 2023 compared to 2022?
- What are the monthly growth trends and seasonality patterns?
- Which product categories drive the most revenue?
- What are our top-performing geographic markets?
- How do delivery times impact customer satisfaction?

### Analysis Configuration

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import yaml
import warnings
from datetime import datetime

# Import custom modules
from data_loader import load_and_prepare_data, DataLoader
from business_metrics import calculate_comprehensive_metrics, BusinessMetricsCalculator

# Configure display and warnings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries imported successfully")
print(f"Analysis run time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Load configuration
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

# Extract key parameters
CURRENT_YEAR = config['analysis']['current_year']
COMPARISON_YEAR = config['analysis']['comparison_year']
DATA_DIR = config['data']['data_directory']

print(f"📊 Analysis Configuration:")
print(f"   Current Year: {CURRENT_YEAR}")
print(f"   Comparison Year: {COMPARISON_YEAR}")
print(f"   Data Directory: {DATA_DIR}")
print(f"   Delivered Orders Only: {config['data']['delivered_only']}")

## 2. Data Dictionary

### Key Business Terms and Metrics

| Term | Definition | Business Importance |
|------|------------|--------------------|
| **Revenue** | Total monetary value from delivered orders | Primary business performance indicator |
| **Average Order Value (AOV)** | Average total value per order | Indicates customer spending behavior |
| **Year-over-Year Growth** | Percentage change compared to previous year | Measures business growth trajectory |
| **Month-over-Month Growth** | Percentage change from previous month | Identifies seasonal trends and momentum |
| **Delivery Speed** | Days between order placement and delivery | Key customer experience metric |
| **Review Score** | Customer satisfaction rating (1-5) | Direct measure of customer satisfaction |

### Dataset Structure

- **orders_dataset.csv**: Order information including timestamps and status
- **order_items_dataset.csv**: Individual items within orders with pricing
- **customers_dataset.csv**: Customer demographics and geographic data
- **products_dataset.csv**: Product catalog with categories
- **order_reviews_dataset.csv**: Customer reviews and ratings
- **order_payments_dataset.csv**: Payment method information

## 3. Data Loading & Validation

Loading and validating e-commerce datasets with quality checks and preprocessing.

In [ ]:
# Load and prepare comprehensive dataset
print("🔄 Loading and preparing data...")

sales_data = load_and_prepare_data(
    data_dir=DATA_DIR,
    years=[CURRENT_YEAR, COMPARISON_YEAR] if COMPARISON_YEAR else [CURRENT_YEAR],
    include_product_info=config['data']['include_product_info'],
    include_customer_info=config['data']['include_customer_info'],
    include_reviews=config['data']['include_review_scores'],
    delivered_only=config['data']['delivered_only']
)

print(f"✅ Data loaded successfully: {len(sales_data):,} records")
print(f"   Date range: {sales_data['order_purchase_timestamp'].min()} to {sales_data['order_purchase_timestamp'].max()}")
print(f"   Years available: {sorted(sales_data['year'].unique())}")

In [ ]:
# Data validation and quality assessment
def display_data_quality_summary(df):
    """Display comprehensive data quality summary."""
    
    print("📋 Data Quality Summary")
    print("=" * 50)
    
    # Basic statistics
    print(f"Total Records: {len(df):,}")
    print(f"Total Columns: {len(df.columns)}")
    print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    # Missing values analysis
    missing_data = df.isnull().sum()
    missing_pct = (missing_data / len(df) * 100).round(2)
    
    if missing_data.sum() > 0:
        print("\n❌ Missing Values:")
        for col in missing_data[missing_data > 0].index:
            print(f"   {col}: {missing_data[col]:,} ({missing_pct[col]}%)")
    else:
        print("\n✅ No missing values detected")
    
    # Key business metrics overview
    print("\n💰 Business Metrics Overview:")
    print(f"   Total Revenue: ${df['price'].sum():,.2f}")
    print(f"   Unique Orders: {df['order_id'].nunique():,}")
    print(f"   Unique Customers: {df['customer_id'].nunique():,}")
    print(f"   Average Order Value: ${df.groupby('order_id')['price'].sum().mean():.2f}")
    
    return df.describe()

# Display data quality summary
data_summary = display_data_quality_summary(sales_data)
data_summary.round(2)

## 4. Revenue & Growth Analysis

### 4.1 Year-over-Year Revenue Performance

Analyzing overall business performance and growth trajectory.

In [ ]:
# Calculate comprehensive business metrics
print("📊 Calculating business metrics...")

metrics = calculate_comprehensive_metrics(
    data=sales_data,
    current_year=CURRENT_YEAR,
    comparison_year=COMPARISON_YEAR
)

# Extract revenue metrics for display
revenue_metrics = metrics['revenue_metrics']
order_metrics = metrics['order_metrics']

print("✅ Metrics calculated successfully")

In [ ]:
# Display key revenue insights
def display_revenue_insights(revenue_metrics, order_metrics):
    """Display key revenue and order insights in a formatted manner."""
    
    print("💰 REVENUE PERFORMANCE INSIGHTS")
    print("=" * 60)
    
    # Current year performance
    current_revenue = revenue_metrics['total_revenue']
    print(f"📈 {revenue_metrics['current_year']} Revenue: ${current_revenue:,.2f}")
    
    # Year-over-year comparison
    if 'yoy_growth_percent' in revenue_metrics:
        growth = revenue_metrics['yoy_growth_percent']
        comparison_revenue = revenue_metrics['comparison_revenue']
        
        growth_indicator = "📈" if growth > 0 else "📉" if growth < 0 else "➡️"
        print(f"{growth_indicator} YoY Growth: {growth:+.2f}%")
        print(f"   {revenue_metrics['comparison_year']} Revenue: ${comparison_revenue:,.2f}")
        print(f"   Revenue Change: ${current_revenue - comparison_revenue:+,.2f}")
    
    # Monthly growth trend
    if 'monthly_growth_trend' in revenue_metrics:
        avg_mom_growth = revenue_metrics['monthly_growth_trend']['average_mom_growth']
        trend_indicator = "📈" if avg_mom_growth > 0 else "📉" if avg_mom_growth < 0 else "➡️"
        print(f"{trend_indicator} Avg Monthly Growth: {avg_mom_growth:+.2f}%")
    
    # Order metrics
    print(f"\n🛒 ORDER PERFORMANCE:")
    print(f"   Total Orders: {order_metrics['total_orders']:,}")
    print(f"   Average Order Value: ${order_metrics['average_order_value']:.2f}")
    print(f"   Items per Order: {order_metrics['average_items_per_order']:.1f}")
    
    if 'order_growth_percent' in order_metrics:
        order_growth = order_metrics['order_growth_percent']
        order_indicator = "📈" if order_growth > 0 else "📉" if order_growth < 0 else "➡️"
        print(f"   {order_indicator} Order Growth: {order_growth:+.2f}%")

display_revenue_insights(revenue_metrics, order_metrics)

### 4.2 Monthly Revenue Trends

Visualizing monthly revenue patterns and identifying seasonal trends.

In [ ]:
# Create monthly revenue trend visualization
def create_monthly_revenue_chart(sales_data, current_year, comparison_year=None):
    """Create interactive monthly revenue trend chart."""
    
    # Prepare monthly data
    monthly_data = sales_data.groupby(['year', 'month'])['price'].sum().reset_index()
    monthly_data['date'] = pd.to_datetime(monthly_data[['year', 'month']].assign(day=1))
    
    # Create figure
    fig = go.Figure()
    
    # Add current year line
    current_data = monthly_data[monthly_data['year'] == current_year]
    fig.add_trace(go.Scatter(
        x=current_data['month'],
        y=current_data['price'],
        mode='lines+markers',
        name=f'{current_year}',
        line=dict(color='#1f77b4', width=3),
        marker=dict(size=8)
    ))
    
    # Add comparison year if available
    if comparison_year:
        comparison_data = monthly_data[monthly_data['year'] == comparison_year]
        fig.add_trace(go.Scatter(
            x=comparison_data['month'],
            y=comparison_data['price'],
            mode='lines+markers',
            name=f'{comparison_year}',
            line=dict(color='#ff7f0e', width=2, dash='dash'),
            marker=dict(size=6)
        ))
    
    # Update layout
    fig.update_layout(
        title=f'Monthly Revenue Trends ({current_year}' + (f' vs {comparison_year})' if comparison_year else ')'),
        xaxis_title='Month',
        yaxis_title='Revenue ($)',
        xaxis=dict(tickmode='array', tickvals=list(range(1, 13)), 
                  ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                           'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']),
        yaxis=dict(tickformat='$,.0f'),
        hovermode='x unified',
        width=1000,
        height=500
    )
    
    return fig

# Display monthly revenue chart
monthly_chart = create_monthly_revenue_chart(sales_data, CURRENT_YEAR, COMPARISON_YEAR)
monthly_chart.show()

# Display monthly growth rates
if 'monthly_growth_rates' in revenue_metrics.get('monthly_growth_trend', {}):
    growth_rates = revenue_metrics['monthly_growth_trend']['monthly_growth_rates']
    
    print(f"\n📊 {CURRENT_YEAR} Month-over-Month Growth Rates:")
    for month, growth in growth_rates.items():
        month_name = pd.to_datetime(f'2023-{month:02d}-01').strftime('%B')
        indicator = "📈" if growth > 0 else "📉" if growth < 0 else "➡️"
        print(f"   {indicator} {month_name}: {growth*100:+.1f}%")

## 5. Product Performance Analysis

### 5.1 Revenue by Product Category

Analyzing which product categories drive the most revenue and identifying top performers.

In [ ]:
# Display product performance insights
product_metrics = metrics['product_performance']

if product_metrics:  # Check if product data is available
    print(f"🛍️ PRODUCT PERFORMANCE INSIGHTS ({CURRENT_YEAR})")
    print("=" * 60)
    
    print(f"Total Product Categories: {product_metrics['category_count']}")
    
    # Top 5 categories
    print("\n🏆 Top 5 Product Categories by Revenue:")
    for i, (category, revenue) in enumerate(product_metrics['top_categories'].items(), 1):
        print(f"   {i}. {category.replace('_', ' ').title()}: ${revenue:,.2f}")
else:
    print("⚠️ Product category data not available")

In [ ]:
# Create product category performance visualization
def create_product_category_chart(product_metrics, current_year):
    """Create horizontal bar chart for product category performance."""
    
    if not product_metrics:
        print("⚠️ No product data available for visualization")
        return None
    
    # Prepare data
    category_data = product_metrics['category_performance']
    
    # Convert to DataFrame for easier manipulation
    df = pd.DataFrame(category_data).T
    df = df.sort_values('total_revenue', ascending=True)  # Sort for horizontal bar chart
    
    # Create figure
    fig = go.Figure()
    
    # Add bar chart
    fig.add_trace(go.Bar(
        y=[cat.replace('_', ' ').title() for cat in df.index],
        x=df['total_revenue'],
        orientation='h',
        marker=dict(
            color=df['total_revenue'],
            colorscale='Blues',
            showscale=False
        ),
        text=[f'${val:,.0f}' for val in df['total_revenue']],
        textposition='inside',
        hovertemplate='<b>%{y}</b><br>' +
                      'Revenue: $%{x:,.0f}<br>' +
                      'Share: %{customdata:.1f}%<extra></extra>',
        customdata=df['revenue_share_percent']
    ))
    
    # Update layout
    fig.update_layout(
        title=f'Revenue by Product Category ({current_year})',
        xaxis_title='Revenue ($)',
        yaxis_title='Product Category',
        xaxis=dict(tickformat='$,.0f'),
        width=1000,
        height=600,
        margin=dict(l=200)  # Left margin for category names
    )
    
    return fig

# Display product category chart
if product_metrics:
    category_chart = create_product_category_chart(product_metrics, CURRENT_YEAR)
    if category_chart:
        category_chart.show()

### 5.2 Category Performance Details

Detailed breakdown of product category metrics including revenue share and performance indicators.

In [ ]:
# Create detailed product performance table
if product_metrics:
    category_performance = pd.DataFrame(product_metrics['category_performance']).T
    
    # Format for better display
    category_performance.index = [cat.replace('_', ' ').title() for cat in category_performance.index]
    category_performance = category_performance.sort_values('total_revenue', ascending=False)
    
    # Round numeric columns
    numeric_cols = ['total_revenue', 'avg_item_price', 'revenue_share_percent']
    category_performance[numeric_cols] = category_performance[numeric_cols].round(2)
    
    print("📊 Detailed Category Performance:")
    display(category_performance.head(10))
else:
    print("⚠️ Product performance data not available")

## 6. Geographic Revenue Analysis

### 6.1 Revenue by State

Analyzing geographic distribution of revenue to identify top-performing markets.

In [ ]:
# Display geographic performance insights
geographic_metrics = metrics['geographic_metrics']

if geographic_metrics:  # Check if geographic data is available
    print(f"🌎 GEOGRAPHIC PERFORMANCE INSIGHTS ({CURRENT_YEAR})")
    print("=" * 60)
    
    print(f"Total States/Regions: {geographic_metrics['state_count']}")
    
    # Top 10 states
    print("\n🏆 Top 10 States by Revenue:")
    for i, (state, revenue) in enumerate(list(geographic_metrics['top_states'].items())[:10], 1):
        print(f"   {i:2d}. {state}: ${revenue:,.2f}")
else:
    print("⚠️ Geographic data not available")

In [ ]:
# Create geographic revenue visualization
def create_geographic_revenue_chart(geographic_metrics, current_year):
    """Create choropleth map for geographic revenue distribution."""
    
    if not geographic_metrics:
        print("⚠️ No geographic data available for visualization")
        return None
    
    # Prepare data
    state_data = geographic_metrics['state_performance']
    
    # Convert to DataFrame
    df = pd.DataFrame(state_data).T.reset_index()
    df.rename(columns={'index': 'state'}, inplace=True)
    
    # Create choropleth map
    fig = px.choropleth(
        df,
        locations='state',
        color='total_revenue',
        locationmode='USA-states',
        scope='usa',
        title=f'Revenue by State ({current_year})',
        color_continuous_scale='Blues',
        labels={'total_revenue': 'Revenue ($)'},
        hover_data={
            'total_orders': ':,',
            'unique_customers': ':,',
            'avg_order_value': ':.2f'
        }
    )
    
    # Update layout
    fig.update_layout(
        width=1000,
        height=600,
        geo=dict(
            showframe=False,
            showcoastlines=True,
            projection_type='albers usa'
        )
    )
    
    return fig

# Display geographic chart
if geographic_metrics:
    geo_chart = create_geographic_revenue_chart(geographic_metrics, CURRENT_YEAR)
    if geo_chart:
        geo_chart.show()

### 6.2 State Performance Details

Detailed analysis of top-performing states with key metrics.

In [ ]:
# Create detailed geographic performance table
if geographic_metrics:
    state_performance = pd.DataFrame(geographic_metrics['state_performance']).T
    state_performance = state_performance.sort_values('total_revenue', ascending=False)
    
    # Round numeric columns
    numeric_cols = ['total_revenue', 'avg_order_value', 'revenue_share_percent']
    state_performance[numeric_cols] = state_performance[numeric_cols].round(2)
    
    print("📊 Top 15 States Performance Details:")
    display(state_performance.head(15))
else:
    print("⚠️ Geographic performance data not available")

## 7. Customer Experience Analysis

### 7.1 Delivery Performance

Analyzing delivery speed and its impact on customer satisfaction.

In [ ]:
# Display customer experience insights
cx_metrics = metrics['customer_experience']

if cx_metrics:  # Check if customer experience data is available
    print(f"📦 CUSTOMER EXPERIENCE INSIGHTS ({CURRENT_YEAR})")
    print("=" * 60)
    
    # Delivery performance
    if 'delivery_performance' in cx_metrics:
        delivery = cx_metrics['delivery_performance']
        print(f"🚚 Delivery Performance:")
        print(f"   Average Delivery Time: {delivery['average_delivery_days']:.1f} days")
        print(f"   Median Delivery Time: {delivery['median_delivery_days']:.1f} days")
        print(f"   95th Percentile: {delivery['delivery_percentiles']['95th']:.1f} days")
    
    # Review performance
    if 'review_performance' in cx_metrics:
        reviews = cx_metrics['review_performance']
        print(f"\n⭐ Customer Satisfaction:")
        print(f"   Average Review Score: {reviews['average_review_score']:.2f}/5.0")
        print(f"   Total Reviews: {reviews['total_reviews']:,}")
    
    # Delivery speed distribution
    if 'delivery_speed_distribution' in cx_metrics:
        speed_dist = cx_metrics['delivery_speed_distribution']
        print(f"\n📊 Delivery Speed Distribution:")
        for category, percentage in speed_dist.items():
            print(f"   {category}: {percentage*100:.1f}%")
    
    # Delivery satisfaction correlation
    if 'delivery_satisfaction_correlation' in cx_metrics:
        correlation = cx_metrics['delivery_satisfaction_correlation']
        print(f"\n📈 Review Score by Delivery Speed:")
        for speed, score in correlation.items():
            print(f"   {speed}: {score:.2f}/5.0")
else:
    print("⚠️ Customer experience data not available")

### 7.2 Customer Satisfaction Visualizations

Visual analysis of review scores and delivery performance.

In [ ]:
# Create customer satisfaction visualizations
def create_customer_satisfaction_charts(cx_metrics, current_year):
    """Create comprehensive customer satisfaction visualizations."""
    
    if not cx_metrics:
        print("⚠️ No customer experience data available")
        return None
    
    # Create subplot figure
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Review Score Distribution',
            'Delivery Speed Distribution', 
            'Review Score by Delivery Speed',
            'Delivery Time Percentiles'
        ),
        specs=[[{"type": "bar"}, {"type": "pie"}],
               [{"type": "bar"}, {"type": "bar"}]]
    )
    
    # 1. Review Score Distribution
    if 'review_performance' in cx_metrics:
        review_dist = cx_metrics['review_performance']['review_distribution']
        scores = list(review_dist.keys())
        percentages = [v * 100 for v in review_dist.values()]
        
        fig.add_trace(
            go.Bar(x=scores, y=percentages, name='Review Scores',
                  marker_color='lightblue'),
            row=1, col=1
        )
    
    # 2. Delivery Speed Distribution
    if 'delivery_speed_distribution' in cx_metrics:
        speed_dist = cx_metrics['delivery_speed_distribution']
        
        fig.add_trace(
            go.Pie(labels=list(speed_dist.keys()), 
                  values=[v * 100 for v in speed_dist.values()],
                  name='Delivery Speed'),
            row=1, col=2
        )
    
    # 3. Review Score by Delivery Speed
    if 'delivery_satisfaction_correlation' in cx_metrics:
        correlation = cx_metrics['delivery_satisfaction_correlation']
        
        fig.add_trace(
            go.Bar(x=list(correlation.keys()), 
                  y=list(correlation.values()),
                  name='Satisfaction by Speed',
                  marker_color='lightcoral'),
            row=2, col=1
        )
    
    # 4. Delivery Time Percentiles
    if 'delivery_performance' in cx_metrics:
        delivery = cx_metrics['delivery_performance']
        percentiles = delivery['delivery_percentiles']
        
        fig.add_trace(
            go.Bar(x=list(percentiles.keys()), 
                  y=list(percentiles.values()),
                  name='Delivery Percentiles',
                  marker_color='lightgreen'),
            row=2, col=2
        )
    
    # Update layout
    fig.update_layout(
        title_text=f"Customer Experience Dashboard ({current_year})",
        width=1200,
        height=800,
        showlegend=False
    )
    
    # Update axes labels
    fig.update_xaxes(title_text="Review Score", row=1, col=1)
    fig.update_yaxes(title_text="Percentage (%)", row=1, col=1)
    
    fig.update_xaxes(title_text="Delivery Category", row=2, col=1)
    fig.update_yaxes(title_text="Avg Review Score", row=2, col=1)
    
    fig.update_xaxes(title_text="Percentile", row=2, col=2)
    fig.update_yaxes(title_text="Days", row=2, col=2)
    
    return fig

# Display customer satisfaction charts
if cx_metrics:
    satisfaction_chart = create_customer_satisfaction_charts(cx_metrics, CURRENT_YEAR)
    if satisfaction_chart:
        satisfaction_chart.show()

## 8. Executive Summary & Key Insights

### 8.1 Key Performance Indicators Summary

In [ ]:
# Generate executive summary
def generate_executive_summary(metrics, current_year, comparison_year=None):
    """Generate comprehensive executive summary of business performance."""
    
    print("" + "=" * 80)
    print(f"📊 EXECUTIVE SUMMARY - {current_year} BUSINESS PERFORMANCE")
    print("" + "=" * 80)
    
    # Revenue Performance
    revenue = metrics['revenue_metrics']
    orders = metrics['order_metrics']
    
    print(f"\n💰 FINANCIAL PERFORMANCE")
    print(f"   • Total Revenue: ${revenue['total_revenue']:,.2f}")
    print(f"   • Total Orders: {orders['total_orders']:,}")
    print(f"   • Average Order Value: ${orders['average_order_value']:.2f}")
    
    if comparison_year and 'yoy_growth_percent' in revenue:
        growth_status = "Growth 📈" if revenue['yoy_growth_percent'] > 0 else "Decline 📉"
        print(f"   • YoY {growth_status}: {revenue['yoy_growth_percent']:+.2f}%")
    
    # Product Performance
    if metrics['product_performance']:
        products = metrics['product_performance']
        top_category = list(products['top_categories'].keys())[0]
        print(f"\n🛍️ PRODUCT PERFORMANCE")
        print(f"   • Top Category: {top_category.replace('_', ' ').title()}")
        print(f"   • Categories Analyzed: {products['category_count']}")
    
    # Geographic Performance
    if metrics['geographic_metrics']:
        geo = metrics['geographic_metrics']
        top_state = list(geo['top_states'].keys())[0]
        print(f"\n🌎 GEOGRAPHIC PERFORMANCE")
        print(f"   • Top State: {top_state}")
        print(f"   • Active States: {geo['state_count']}")
    
    # Customer Experience
    if metrics['customer_experience']:
        cx = metrics['customer_experience']
        
        print(f"\n📦 CUSTOMER EXPERIENCE")
        if 'delivery_performance' in cx:
            avg_delivery = cx['delivery_performance']['average_delivery_days']
            print(f"   • Average Delivery Time: {avg_delivery:.1f} days")
        
        if 'review_performance' in cx:
            avg_rating = cx['review_performance']['average_review_score']
            print(f"   • Average Customer Rating: {avg_rating:.2f}/5.0")
    
    print(f"\n" + "=" * 80)
    print(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)

# Generate and display executive summary
generate_executive_summary(metrics, CURRENT_YEAR, COMPARISON_YEAR)

### 8.2 Strategic Recommendations

Based on the analysis, here are key strategic recommendations:

In [ ]:
# Generate strategic recommendations
def generate_recommendations(metrics, current_year, comparison_year=None):
    """Generate strategic recommendations based on analysis results."""
    
    recommendations = []
    
    # Revenue-based recommendations
    revenue = metrics['revenue_metrics']
    if comparison_year and 'yoy_growth_percent' in revenue:
        if revenue['yoy_growth_percent'] < 0:
            recommendations.append(
                "🎯 **Revenue Recovery**: Implement targeted marketing campaigns to reverse negative growth trend"
            )
        elif revenue['yoy_growth_percent'] < 5:
            recommendations.append(
                "📈 **Growth Acceleration**: Explore new customer acquisition channels to boost growth above 5%"
            )
    
    # Monthly growth recommendations
    if 'monthly_growth_trend' in revenue:
        avg_growth = revenue['monthly_growth_trend']['average_mom_growth']
        if avg_growth < 0:
            recommendations.append(
                "📊 **Seasonal Strategy**: Develop seasonal promotions to address monthly revenue volatility"
            )
    
    # Product recommendations
    if metrics['product_performance']:
        products = metrics['product_performance']
        if products['category_count'] > 5:
            recommendations.append(
                "🛍️ **Category Focus**: Consider focusing marketing spend on top 3 performing categories"
            )
    
    # Customer experience recommendations
    if metrics['customer_experience']:
        cx = metrics['customer_experience']
        
        if 'delivery_performance' in cx:
            avg_delivery = cx['delivery_performance']['average_delivery_days']
            if avg_delivery > 7:
                recommendations.append(
                    "🚚 **Delivery Optimization**: Improve logistics to reduce delivery time below 7 days"
                )
        
        if 'review_performance' in cx:
            avg_rating = cx['review_performance']['average_review_score']
            if avg_rating < 4.0:
                recommendations.append(
                    "⭐ **Customer Satisfaction**: Implement quality improvement program to boost ratings above 4.0"
                )
        
        if 'delivery_satisfaction_correlation' in cx:
            correlation = cx['delivery_satisfaction_correlation']
            fast_delivery = correlation.get('1-3 days', 0)
            slow_delivery = correlation.get('8+ days', 0)
            
            if fast_delivery > slow_delivery:
                recommendations.append(
                    "⚡ **Fast Delivery Premium**: Consider premium delivery options to improve satisfaction"
                )
    
    # Geographic recommendations
    if metrics['geographic_metrics']:
        geo = metrics['geographic_metrics']
        if geo['state_count'] > 10:
            recommendations.append(
                "🌎 **Geographic Expansion**: Evaluate market penetration in underperforming states"
            )
    
    return recommendations

# Generate and display recommendations
recommendations = generate_recommendations(metrics, CURRENT_YEAR, COMPARISON_YEAR)

print("🎯 STRATEGIC RECOMMENDATIONS")
print("=" * 50)

if recommendations:
    for i, rec in enumerate(recommendations, 1):
        print(f"\n{i}. {rec}")
else:
    print("\n✅ Current performance metrics are within acceptable ranges.")
    print("   Continue monitoring trends and maintain current strategies.")

print(f"\n" + "=" * 50)
print("💡 **Next Steps**: Prioritize recommendations based on business impact and implementation feasibility")

### 8.3 Analysis Configuration Summary

Review of analysis parameters and data coverage for transparency and reproducibility.

In [ ]:
# Display analysis configuration summary
print("⚙️ ANALYSIS CONFIGURATION SUMMARY")
print("=" * 50)

print(f"📅 Analysis Period: {CURRENT_YEAR}" + (f" vs {COMPARISON_YEAR}" if COMPARISON_YEAR else ""))
print(f"📊 Total Records Analyzed: {len(sales_data):,}")
print(f"🗂️ Data Directory: {DATA_DIR}")
print(f"✅ Delivered Orders Only: {config['data']['delivered_only']}")

print(f"\n📈 Metrics Calculated:")
print(f"   • Revenue & Growth Analysis")
print(f"   • Order Volume & Value Analysis")
if metrics['product_performance']:
    print(f"   • Product Category Performance")
if metrics['geographic_metrics']:
    print(f"   • Geographic Revenue Distribution")
if metrics['customer_experience']:
    print(f"   • Customer Experience & Satisfaction")

print(f"\n🔧 Enhancement Options:")
print(f"   • Product Info: {config['data']['include_product_info']}")
print(f"   • Customer Geography: {config['data']['include_customer_info']}")
print(f"   • Review Scores: {config['data']['include_review_scores']}")

print(f"\n📋 Data Quality:")
validation_summary = {
    'Total Revenue': f"${sales_data['price'].sum():,.2f}",
    'Unique Orders': f"{sales_data['order_id'].nunique():,}",
    'Unique Customers': f"{sales_data['customer_id'].nunique():,}",
    'Date Range': f"{sales_data['order_purchase_timestamp'].min().strftime('%Y-%m-%d')} to {sales_data['order_purchase_timestamp'].max().strftime('%Y-%m-%d')}"
}

for metric, value in validation_summary.items():
    print(f"   • {metric}: {value}")

print(f"\n" + "=" * 50)
print(f"🔄 Analysis Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📧 For questions about this analysis, review the configuration in config.yaml")

---

## Analysis Complete ✅

This comprehensive e-commerce business analysis provides insights into:

- **Revenue Performance**: Year-over-year and monthly trend analysis
- **Product Strategy**: Category performance and market share analysis  
- **Geographic Opportunities**: State-level revenue distribution
- **Customer Experience**: Delivery performance and satisfaction correlation
- **Strategic Recommendations**: Data-driven business improvement suggestions

### Next Steps

1. **Review Recommendations**: Prioritize strategic recommendations based on business impact
2. **Customize Analysis**: Modify `config.yaml` to analyze different time periods or metrics
3. **Export Results**: Save charts and data using the export functionality
4. **Schedule Updates**: Set up regular analysis runs to monitor performance trends

### Support

- Configuration options are available in `config.yaml`
- Custom metrics can be added to `business_metrics.py`
- Data processing can be modified in `data_loader.py`

---

*Generated using the E-commerce Business Analytics Framework*